# News-Tracker: woechentlicher News-Digest fuer das Depot

1. **Sammeln (Python):** Finnhub und SEC EDGAR 8-K (US-Werte), Yahoo Finance (auch Europa), Google-News-RSS (deutsch + englisch). Relevanzfilter per Name/Ticker, Dubletten raus, Speicher `news/news_store.jsonl` (jede Meldung wird nur einmal klassifiziert).
2. **Klassifizieren (guenstiges Modell, gebuendelt):** Kategorie, Richtung, Wesentlichkeit 1-3, betroffene Investment-These (aus dem Annual Report Agent / Thesen-Protokoll).
3. **Marktreaktion (Python):** Ueberrendite ggue. MSCI ACWI am Tag 0 und Tag 0..+1 (EUR), Volumen ggue. 20-Tage-Median.
4. **Stories und Signale:** wesentliche Meldungen werden zu Stories gebuendelt (Status neu / eskaliert / beruhigt), Querschnittsthemen ueber mehrere Positionen; Python berechnet News-Volumen ggue. Schnitt, Sentiment-Trend, Divergenz News vs. Kurs, Analysten-Revisionen.
5. **Digest (Agent):** Signale, Stories, Thesen-Alarme, Notizen je Position, Analystenaktionen (getrennt von den News), Faktencheck, PDF.

Watchlist: aktuelle Bestaende aus den Transaktions-Exporten -> `portfolios/watchlist.csv` (von Hand anpassbar: Suchbegriff `query`, `track` = no zum Ausschliessen, ETFs sind automatisch no). Alles bleibt lokal (CSV gitignored).

In [ ]:
# 1) Pfade und Imports
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
FR = ROOT / "finrobot_autogen"
sys.path.insert(0, str(AIF))

import pandas as pd
from finrobot.utils import register_keys_from_json
from lib import news_tracker as nt

In [ ]:
# 2) Modelle und Schluessel: Klassifizierung (guenstig, viele Meldungen) und Digest (schreibt den Text)
WRITER_MODEL = "anthropic/claude-haiku-5.5"
CLASSIFIER_MODEL = "anthropic/claude-haiku-5.5"     # z.B. "google/gemini-4-neon" zum Vergleich

base = json.load(open(FR / "OAI_CONFIG_LIST", encoding="utf-8"))
def cfg(model, **kw):
    entry = next((e for e in base if e["model"] == model), None) or dict(base[0], model=model)
    c = {"config_list": [entry], "timeout": 300, "temperature": 0.3, "max_tokens": 8000, **kw}
    if model.startswith("google/"):
        c["extra_body"] = {"reasoning": {"effort": "low"}}
    return c
llm_config = cfg(WRITER_MODEL, parallel_tool_calls=False)
classifier_config = cfg(CLASSIFIER_MODEL, temperature=0)
register_keys_from_json(str(FR / "config_api_keys"))

In [ ]:
# 3) Watchlist aus den aktuellen Bestaenden (einmalig bzw. nach Kaeufen/Verkaeufen neu erzeugen)
TRANSACTION_FILES = [AIF / "portfolios" / f for f in ("traderepublic.csv", "ScalableCapital.csv", "Smartbroker1.csv",
                                                      "Smartbroker2.csv")]
HOLDINGS_FILES = []        # nur Positions-Exporte ohne Kaufhistorie
nt.build_watchlist(TRANSACTION_FILES, holdings_files=HOLDINGS_FILES)          # bestehende Eintraege (query/track) bleiben erhalten
pd.DataFrame(nt.load_watchlist(only_tracked=False))

## Optional, einmalig: Rueckblick (Backfill)
Signale (News-Volumen, Sentiment-Trend) brauchen Historie. Finnhub liefert fuer US-Werte bis zu ein Jahr rueckwirkend; Yahoo und Google nur aktuelle Meldungen (europaeische Werte bekommen erst ab jetzt Historie). Kosten grob: Positionen x Wochen x bis zu 15 Meldungen klassifizieren, bei Haiku fuer 5 Positionen und 8 Wochen ca. 0,1-0,2 Mio. Tokens.

In [ ]:
# 3b) Backfill (nur einmal ausfuehren; danach reicht der woechentliche Lauf)
RUN_BACKFILL = False
if RUN_BACKFILL:
    wl_bf = nt.top_watchlist(TRANSACTION_FILES, n=5, holdings_files=HOLDINGS_FILES)      # oder nt.load_watchlist() fuer alle
    nt.backfill(wl_bf, classifier_config, weeks=8, per_week=15, stories=True)

In [ ]:
# 4) Sammeln, klassifizieren, Marktreaktion, Digest (Standard: letzte 7 Tage)
DAYS = 7
TICKERS = ["GOOG", "AMZN", "MA"]   # Musterportfolio-Titel; None = Top-N aus den Broker-Dateien (Zelle 3 vorher ausfuehren)
TOP_N = 5                          # nur wenn TICKERS = None: groesste Positionen aus dem Depot
if TICKERS:
    watchlist = nt.watchlist_from_tickers(TICKERS)
else:
    watchlist = nt.top_watchlist(TRANSACTION_FILES, n=TOP_N, holdings_files=HOLDINGS_FILES) if TOP_N else None
print("Watchlist:", [w["ticker"] for w in watchlist] if watchlist else "ganze Watchlist")
save_dir = f"../reports/news_{WRITER_MODEL.split('/')[-1]}"
state = nt.run_news_tracker(llm_config, classifier_config, watchlist=watchlist, days=DAYS, use_google=True,
                            save_dir=save_dir)

In [ ]:
# 5) Uebersicht der Woche (das sieht auch der Agent)
print(nt.overview_text(state["items"], state["stats"], state["watchlist"]))

## Evaluation (fuer den Kurs-Report)
- **Klassifizierungsguete:** Stichprobe exportieren, in `news/label_sample.csv` die Spalten `label_*` von Hand ausfuellen (relevant: True/False, category, direction, materiality), dann auswerten.
- **Marktuebereinstimmung:** Anteil der wesentlichen Meldungen, bei denen die Richtung der Klassifizierung zum Vorzeichen der Ueberrendite passt (Bewegungen ueber 1 %).

In [ ]:
# 6) Stichprobe zum Labeln exportieren (nach einigen Wochen Sammeln sinnvoll)
nt.export_label_sample(n=100)

In [ ]:
# 7) Auswertung nach dem Labeln
nt.evaluate_labels()